# EVE Internal Diagnostics

Visualise the internal dynamics of the EVE optimizer during training:
selection temperature, offspring fitness and weights, direction norms,
and pairwise / combined cosine similarities.

**Model:** ResNet-18 on CIFAR-10.

In [ ]:
import os, sys, subprocess

BRANCH = "full-batch-probing"

if os.path.exists("/content"):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH,
         "https://github.com/SattamAltwaim/EVE.git", "/content/EVE"],
        capture_output=True, check=False,
    )
    if "/content/EVE" not in sys.path:
        sys.path.insert(0, "/content/EVE")
else:
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.exists(os.path.join(parent, "eve_optimizer")) and parent not in sys.path:
        sys.path.insert(0, parent)

from eve_optimizer import EVE
print("EVE imported successfully")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

## Configuration

In [ ]:
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 128
VAL_BATCH   = 512
NUM_WORKERS = 0
LR          = 1e-3
K           = 4
EPOCHS      = 15
SEED        = 42

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Config : batch={BATCH_SIZE}  val_batch={VAL_BATCH}  lr={LR}  K={K}  epochs={EPOCHS}")

## Data

In [ ]:
train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
val_tf = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_ds = torchvision.datasets.CIFAR10(
    root="/tmp/cifar10", train=True, download=True, transform=train_tf)
val_ds = torchvision.datasets.CIFAR10(
    root="/tmp/cifar10", train=False, download=True, transform=val_tf)

train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"), drop_last=True)
val_dl = torch.utils.data.DataLoader(
    val_ds, batch_size=VAL_BATCH, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

print(f"Train batches: {len(train_dl)}  |  Val samples: {len(val_ds)}")

## Model

In [ ]:
def make_model():
    m = torchvision.models.resnet18(weights=None, num_classes=10)
    return m.to(DEVICE)

## Training with diagnostics

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, preds, targets = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out = model(xb)
        total_loss += criterion(out, yb).item() * len(yb)
        preds.extend(out.argmax(1).cpu().tolist())
        targets.extend(yb.cpu().tolist())
    n = len(targets)
    acc = sum(p == t for p, t in zip(preds, targets)) / n
    macro_f1 = f1_score(targets, preds, average="macro", zero_division=0)
    return total_loss / n, acc, macro_f1

In [ ]:
def train_eve_with_diagnostics():
    torch.manual_seed(SEED)
    model = make_model()
    loss_fn = nn.CrossEntropyLoss()
    opt = EVE(model.parameters(), lr=LR, K=K, record_diagnostics=True)

    hist = dict(train_loss=[], val_loss=[], val_acc=[], val_f1=[])

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss, n_batches = 0.0, 0

        pbar = tqdm(train_dl, desc=f"Ep {epoch:2d}/{EPOCHS}", leave=False,
                    dynamic_ncols=True)
        for xb, yb in pbar:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step(model=model, loss_fn=loss_fn, data=(xb, yb),
                     current_loss=loss.item())
            epoch_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=f"{loss.item():.3f}")

        tr_loss = epoch_loss / n_batches
        v_loss, v_acc, v_f1 = evaluate(model, val_dl)

        hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(v_loss)
        hist["val_acc"].append(v_acc)
        hist["val_f1"].append(v_f1)

        print(f"  Ep {epoch:2d}  train_loss={tr_loss:.4f}  "
              f"val_loss={v_loss:.4f}  val_acc={v_acc:.4f}  val_f1={v_f1:.4f}")

    return opt, hist

In [ ]:
opt, hist = train_eve_with_diagnostics()

## Extract diagnostics into arrays

In [ ]:
diag = opt._diagnostics

steps     = np.array([d["step"] for d in diag])
beta_sel  = np.array([d["beta_sel"] for d in diag])
fitness   = np.array([d["fitness"] for d in diag])    # (S, K)
weights   = np.array([d["weights"] for d in diag])    # (S, K)
dir_norms = np.array([d["dir_norms"] for d in diag])  # (S, K)

pair_keys = sorted(diag[0]["cos_pairs"].keys())
cos_pairs = {k: np.array([d["cos_pairs"][k] for d in diag]) for k in pair_keys}

cos_to_combined = np.array([d["cos_to_combined"] for d in diag])  # (S, K)

n_offspring = fitness.shape[1]
labels = [f"d{i+1}" for i in range(n_offspring)]

print(f"Collected {len(diag)} diagnostic steps across {EPOCHS} epochs")
print(f"Offspring count: {n_offspring}")

In [ ]:
COLORS = ["#E91E63", "#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#00BCD4"]
PAIR_COLORS = ["#E91E63", "#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#795548"]

## Selection temperature (beta_sel) per step

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, beta_sel, color=COLORS[0], linewidth=1.2)
ax.set_xlabel("Global step")
ax.set_ylabel("beta_sel")
ax.set_title("Selection Temperature per Step")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Offspring fitness

In [ ]:
print("Offspring fitness summary (L_base - L_k):")
print(f"{'':>6s}", end="")
for lbl in labels:
    print(f"  {lbl:>10s}", end="")
print()
for stat_name, fn in [("mean", np.mean), ("std", np.std),
                       ("min", np.min), ("max", np.max)]:
    print(f"{stat_name:>6s}", end="")
    for k in range(n_offspring):
        print(f"  {fn(fitness[:, k]):10.5f}", end="")
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for k in range(n_offspring):
    ax.plot(steps, fitness[:, k], color=COLORS[k], label=labels[k],
            linewidth=1.0, alpha=0.85)
ax.set_xlabel("Global step")
ax.set_ylabel("Fitness (L_base - L_k)")
ax.set_title("Offspring Fitness per Step")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Offspring selection weights

In [ ]:
print("Offspring weight summary:")
print(f"{'':>6s}", end="")
for lbl in labels:
    print(f"  {lbl:>10s}", end="")
print()
for stat_name, fn in [("mean", np.mean), ("std", np.std),
                       ("min", np.min), ("max", np.max)]:
    print(f"{stat_name:>6s}", end="")
    for k in range(n_offspring):
        print(f"  {fn(weights[:, k]):10.5f}", end="")
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for k in range(n_offspring):
    ax.plot(steps, weights[:, k], color=COLORS[k], label=labels[k],
            linewidth=1.0, alpha=0.85)
ax.set_xlabel("Global step")
ax.set_ylabel("Softmax weight")
ax.set_title("Offspring Selection Weights per Step")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Direction L2 norms

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for k in range(n_offspring):
    ax.plot(steps, dir_norms[:, k], color=COLORS[k], label=labels[k],
            linewidth=1.0, alpha=0.85)
ax.set_xlabel("Global step")
ax.set_ylabel("L2 norm")
ax.set_title("Direction L2 Norms per Offspring")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cosine similarity between offspring direction pairs

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for i, key in enumerate(pair_keys):
    ax.plot(steps, cos_pairs[key], color=PAIR_COLORS[i % len(PAIR_COLORS)],
            label=key, linewidth=1.0, alpha=0.85)
ax.set_xlabel("Global step")
ax.set_ylabel("Cosine similarity")
ax.set_title("Pairwise Cosine Similarity Between Offspring Directions")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cosine similarity of each offspring to the combined direction

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for k in range(n_offspring):
    ax.plot(steps, cos_to_combined[:, k], color=COLORS[k], label=labels[k],
            linewidth=1.0, alpha=0.85)
ax.set_xlabel("Global step")
ax.set_ylabel("Cosine similarity to combined")
ax.set_title("Offspring Direction vs Weighted Combined Direction")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()